Самостоятельное задание:
Необходимо выбрать одну из automl моделей и обучить задачу регрессии.

Напомню, что за данные

### Данные

Каждая строка — заказ доставки еды за 2025 год.
В данных есть характеристики заказа, курьера, ресторана, погоды и дорожной ситуации.

**Основные признаки:**

* `order_datetime` — время заказа (с января по октябрь 2025).
* `distance_km`, `items_count`, `subtotal`, `prep_time_min` — особенности заказа.
* `courier_experience_yrs`, `courier_rating`, `restaurant_rating`.
* `weather`, `traffic`, `area`, `temperature_c`, `holiday`, `surge_coef`.
* **`delivery_time_min`** — целевая переменная (время доставки в минутах).



In [17]:
import pandas as pd

url = "https://raw.githubusercontent.com/totiela/stepik_simulator_ds/main/lessons/lesson_4_1/4_1_homework_dataset.csv"
df  = pd.read_csv(url)

df.sample(3)

,order_id,order_datetime,distance_km,items_count,subtotal,prep_time_min,courier_experience_yrs,courier_rating,restaurant_rating,weather,traffic,is_rain,is_snow,temperature_c,weekday,hour,surge_coef,area,holiday,delivery_time_min
38455,O038456,2025-04-08 09:50:24.292526150,2.36,4,45.87,23.45,6.36,4.69,4.48,wind,low,0,0,18.1,1,9,1.00,South,0,39.66
42177,O042178,2025-04-17 11:53:02.376325767,3.26,3,35.13,14.26,1.92,4.59,4.46,rain,medium,1,0,11.2,3,11,1.05,North,0,52.51
74134,O074135,2025-07-07 18:57:11.555577094,0.84,1,19.07,14.50,0.52,4.52,4.44,cloudy,medium,0,0,7.0,0,18,1.20,West,0,35.22


Разделим данные на train и test (out-of-time). train - выборка для обучения, oot - исключительно для финальной валидации.

In [18]:
train_df = df[df['order_datetime'] < '2025-09-01']
oot_df = df[df['order_datetime'] >= '2025-09-01']

len(train_df), len(oot_df)

(95946, 24054)

In [19]:
target = 'delivery_time_min'
cols_to_drop = ['order_id', 'order_datetime']

Далее обучите любую из automl моделей

https://lightautoml.readthedocs.io/en/v.0.4.0/pages/tutorials/Tutorial_1_basics.html


In [20]:
# чтобы в colab установить AutoGluon - расскоментируй скрип
!pip install autogluon -q --extra-index-url https://download.pytorch.org/whl/cpu

In [21]:
# чтобы в colab установить LightAutoML - расскоментируй скрипт ниже а затем перезапусти ядро.
# !pip install numpy==1.26.4
# !pip install -U 'lightautoml[all]'

In [22]:
# ваш код

После обучения посмотрите метрику MAE на train и oot выборках -> внесите ваше значение метрики на OOT в степик.

In [23]:
from sklearn.metrics import mean_absolute_error

# Ваш код

train_data = train_df.drop(cols_to_drop, axis=1)
oot_data = oot_df.drop(cols_to_drop, axis=1)

In [24]:
from autogluon.tabular import TabularDataset, TabularPredictor

In [25]:
time_limit = 180  # время в секундах
metric = 'mae'  # валдиационная метрика
predictor = TabularPredictor(
    target,
    eval_metric=metric).fit(train_data,
                            time_limit=time_limit,
                            presets='high_quality' # самый простой режим
                            )

No path specified. Models will be saved in: "AutogluonModels/ag-20261025_163312"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.13.16
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.11.0+cu130
CUDA Version:       13.0
GPU Memory:         GPU 0: 14.56/14.56 GB
Total GPU Memory:   Free: 14.56 GB, Allocated: 0.00 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       10.82 GB / 12.67 GB (85.4%)
Disk Space Avail:   68.46 GB / 112.64 GB (60.8%)
Presets specified: ['high_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Note: `save_bag_folds=False`! This will

In [26]:
predictor.leaderboard()

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L3,-2.937941,mean_absolute_error,19.940503,136.151660,0.003841,0.099529,3,False,4
1,LightGBMXT_BAG_L1,-2.940026,mean_absolute_error,15.482754,90.809442,15.482754,90.809442,1,False,1
2,WeightedEnsemble_L2,-2.940026,mean_absolute_error,15.485146,90.816978,0.002392,0.007537,2,False,2
3,LightGBMXT_BAG_L2,-2.961147,mean_absolute_error,19.936662,136.052131,4.453908,45.242689,2,False,3
4,WeightedEnsemble_L3_FULL,NaN,mean_absolute_error,NaN,11.860335,NaN,0.099529,3,True,8
5,WeightedEnsemble_L2_FULL,NaN,mean_absolute_error,NaN,7.900008,NaN,0.007537,2,True,6
6,LightGBMXT_BAG_L2_FULL,NaN,mean_absolute_error,NaN,11.760806,NaN,3.868335,2,True,7
7,LightGBMXT_BAG_L1_FULL,NaN,mean_absolute_error,NaN,7.892471,NaN,7.892471,1,True,5


In [28]:
train_preds = predictor.predict(train_data)
oot_preds = predictor.predict(oot_data)

In [39]:
print(f'train score: {mean_absolute_error(train_data[target].values, train_preds.values)}')
print(f'oot score: {mean_absolute_error(oot_data[target].values, oot_preds.values)}')

train score: 2.7740219948954907
oot score: 2.9568307703820174
